In [2]:
import pandas as pd

csv_path = "outputs/standard_4x4/aggregated/downstream_aggregated_mean_std.csv"
tex_path = "outputs/standard_4x4/aggregated/downstream_aggregated_mean_std.tex"

df = pd.read_csv(csv_path)

# Convert "+-" to LaTeX \pm and wrap metric cells in math mode
metric_cols = [c for c in df.columns if c not in ["Policy", "Task"]]
for c in metric_cols:
    df[c] = (
        df[c]
        .str.replace("+-", r"\\pm", regex=False)
        .apply(lambda s: f"${s}$")
    )

latex = df.to_latex(
    index=False,
    escape=False,      # needed so \pm and $...$ are not escaped
    column_format="ll" + "c" * len(metric_cols),
    caption="FrozenLake downstream results (mean $\\pm$ std across seeds).",
    label="tab:frozenlake_downstream",
)

# with open(tex_path, "w") as f:
#     f.write(latex)

# print(f"Saved: {tex_path}")

In [4]:
#!/usr/bin/env python3
from __future__ import annotations

from pathlib import Path
import pandas as pd


def build_mean_std_latex(
    mean_csv: str,
    std_csv: str,
    out_tex: str,
    decimals: int = 2,
    caption: str = "FrozenLake downstream results (mean $\\pm$ std).",
    label: str = "tab:frozenlake_downstream_mean_std",
) -> None:
    mean_df = pd.read_csv(mean_csv)
    std_df = pd.read_csv(std_csv)

    key_cols = ["Policy", "Task"]
    metric_cols = [c for c in mean_df.columns if c not in key_cols]

    # Basic schema checks
    if list(mean_df.columns) != list(std_df.columns):
        raise ValueError("Mean and std CSV columns do not match.")

    # Merge on keys so rows align safely
    merged = mean_df.merge(
        std_df,
        on=key_cols,
        suffixes=("_mean", "_std"),
        how="inner",
    )

    # Build formatted table: each metric cell is "mean \\pm std"
    out_df = merged[key_cols].copy()
    for col in metric_cols:
        m = merged[f"{col}_mean"].astype(float).round(decimals)
        s = merged[f"{col}_std"].astype(float).round(decimals)
        out_df[col] = [f"${mv:.{decimals}f} \\pm {sv:.{decimals}f}$" for mv, sv in zip(m, s)]

    latex = out_df.to_latex(
        index=False,
        escape=False,  # keep LaTeX math formatting
        caption=caption,
        label=label,
        column_format="ll" + "c" * len(metric_cols),
    )

    Path(out_tex).parent.mkdir(parents=True, exist_ok=True)
    Path(out_tex).write_text(latex, encoding="utf-8")
    print(f"Saved LaTeX table to: {out_tex}")


# if __name__ == "__main__":
base = Path("outputs/standard_4x4/aggregated")
build_mean_std_latex(
    mean_csv=str(base / "downstream_aggregated_mean.csv"),
    std_csv=str(base / "downstream_aggregated_std.csv"),
    out_tex=str(base / "downstream_aggregated_mean_std.tex"),
    decimals=2,
)

Saved LaTeX table to: outputs/standard_4x4/aggregated/downstream_aggregated_mean_std.tex


In [5]:
import pandas as pd
from pathlib import Path

def build_task_latex_tables(
    mean_csv: str,
    std_csv: str,
    out_dir: str,
    decimals: int = 2,
):
    mean_df = pd.read_csv(mean_csv)
    std_df = pd.read_csv(std_csv)
    key_cols = ["Policy", "Task"]
    metric_cols = [c for c in mean_df.columns if c not in key_cols]

    # Merge on keys so rows align safely
    merged = mean_df.merge(
        std_df,
        on=key_cols,
        suffixes=("_mean", "_std"),
        how="inner",
    )

    for task in [1, 2]:
        task_df = merged[merged["Task"] == task]
        out_df = task_df[key_cols[:1]].copy()  # Only Policy column
        for col in metric_cols:
            m = task_df[f"{col}_mean"].astype(float).round(decimals)
            s = task_df[f"{col}_std"].astype(float).round(decimals)
            out_df[col] = [f"${mv:.{decimals}f} \\pm {sv:.{decimals}f}$" for mv, sv in zip(m, s)]

        latex = out_df.to_latex(
            index=False,
            escape=False,
            caption=f"FrozenLake downstream results (Task {task}, mean $\\pm$ std).",
            label=f"tab:frozenlake_downstream_task{task}",
            column_format="l" + "c" * len(metric_cols),
        )

        Path(out_dir).mkdir(parents=True, exist_ok=True)
        out_path = Path(out_dir) / f"downstream_aggregated_task{task}_mean_std.tex"
        out_path.write_text(latex, encoding="utf-8")
        print(f"Saved LaTeX table for Task {task} to: {out_path}")

# if __name__ == "__main__":
base = Path("outputs/standard_4x4/aggregated")
build_task_latex_tables(
    mean_csv=str(base / "downstream_aggregated_mean.csv"),
    std_csv=str(base / "downstream_aggregated_std.csv"),
    out_dir=str(base),
    decimals=2,
)

Saved LaTeX table for Task 1 to: outputs/standard_4x4/aggregated/downstream_aggregated_task1_mean_std.tex
Saved LaTeX table for Task 2 to: outputs/standard_4x4/aggregated/downstream_aggregated_task2_mean_std.tex
